# Week 07 · 변형·합성 프로토타입 미션

하나의 원본에서 자르기·크기 변경·회전을 적용한 레이어 세 개를 만들고, 1000 × 1000 RGBA 캔버스에 합성합니다. `EDIT` 표시가 있는 값만 수정합니다. 마지막 셀의 **WEEK 07 TRANSFORMATION PROTOTYPE COMPLETE**를 확인한 뒤 노트북과 PNG를 제출합니다.

## STEP 0 · 준비와 수업용 원본 생성
이 셀은 실습에 필요한 라이브러리와 6주차 결과물이 없을 때 사용할 대체 이미지를 준비합니다. 수정하지 않고 실행합니다.

In [ ]:
# DO NOT EDIT · 준비 셀
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

EXECUTION_ORDER = [1]
FALLBACK_SOURCE_PATH = "week07_source_poster.png"
LAYER_SIZES = ((420, 420), (340, 340), (260, 260))

def build_fallback_source(path):
    image = Image.new("RGBA", (900, 900), (242, 238, 226, 255))
    draw = ImageDraw.Draw(image)
    ink = (29, 33, 31, 255)
    draw.rectangle((63, 63, 837, 837), outline=ink, width=6)
    draw.rectangle((63, 63, 315, 837), fill=(38, 104, 111, 255))
    draw.ellipse((162, 135, 648, 621), fill=(234, 184, 62, 255), outline=ink, width=6)
    draw.polygon([(522, 108), (792, 387), (486, 531)], fill=(218, 92, 74, 255), outline=ink)
    draw.rectangle((351, 495, 765, 756), fill=(70, 91, 145, 255), outline=ink, width=6)
    for offset in range(0, 306, 50):
        draw.line((90, 648 + offset, 297, 576 + offset), fill=(225, 217, 196, 255), width=7)
    draw.ellipse((414, 549, 603, 738), fill=(133, 177, 151, 255), outline=ink, width=5)
    draw.line((81, 801, 819, 801), fill=ink, width=7)
    image.save(path)

build_fallback_source(FALLBACK_SOURCE_PATH)
print("STEP 0 완료 · 수업용 원본이 준비되었습니다.")
display(Image.open(FALLBACK_SOURCE_PATH))

## STEP 1 · 제출 정보와 출처 기록
학번, 이름, 구성 의도를 작성합니다. 수업용 원본을 사용하면 아래 출처 정보는 유지합니다. 자신의 6주차 PNG를 사용하면 파일을 업로드한 뒤 `source_path`와 출처 다섯 항목을 실제 정보로 바꿉니다.

In [ ]:
# EDIT 1 · 따옴표 안의 내용을 자신의 정보로 바꿉니다.
student_id = "학번을 입력하세요"
student_name = "이름을 입력하세요"
composition_intent = "구성 의도를 20자 이상 작성하세요."
output_filename = f"week07_{student_id}_{student_name}.png"

# EDIT 2 · 기본값은 수업용 원본입니다. 자신의 파일을 쓰면 여섯 값을 모두 수정합니다.
source_choice = "provided"  # provided 또는 own
source_path = FALLBACK_SOURCE_PATH
source_title = "Week 07 Source Poster"
source_creator = "Course-provided asset"
source_url = "Bundled in the Week 07 notebook"
source_license = "Course use / provided asset"
change_description = "자르기, 크기 변경, 회전, 투명도 조절과 레이어 합성"

EXECUTION_ORDER.append(2)
print("제출자:", student_id, student_name)
print("구성 의도:", composition_intent)
print("원본 기록:", source_title, "/", source_creator, "/", source_license)
print("변형 내용:", change_description)

## STEP 2 · 원본 열기, 작업본 복사, 자르기 영역 선택
처음 값은 원본 전체를 선택하므로 마지막 검사에서 통과하지 않습니다. `crop_box`를 `(left, top, right, bottom)` 순서로 수정해 원본보다 작은 영역을 선택합니다.

In [ ]:
# DO NOT EDIT · 원본 열기와 작업본 만들기
assert Path(source_path).exists(), f"파일을 찾을 수 없습니다: {source_path}"
source_bytes_before = Path(source_path).read_bytes()
source_image = Image.open(source_path).convert("RGBA")
working_image = source_image.copy()

# EDIT 3 · 원본보다 작은 유효한 영역으로 바꿉니다.
crop_box = (0, 0, working_image.width, working_image.height)

cropped_preview = working_image.crop(crop_box)
EXECUTION_ORDER.append(3)
print("원본 크기:", source_image.size, "/ 자르기 영역:", crop_box)
print("자른 미리보기 크기:", cropped_preview.size)
display(cropped_preview)

## STEP 3 · 세 레이어 변형하고 합성하기
`angles`, `alphas`, `positions`만 수정합니다. 각도에는 음수와 양수를 모두 사용하고, 알파 값은 최소 두 가지로 구분하며, 위치 세 개는 서로 달라야 합니다.

In [ ]:
# DO NOT EDIT · 한 원본을 하나의 변형 레이어로 만드는 함수
def make_layer(source, crop_box, target_size, angle, alpha):
    layer = source.copy()
    layer = layer.crop(crop_box)
    layer = layer.resize(target_size, Image.Resampling.LANCZOS)
    layer.putalpha(alpha)
    layer = layer.rotate(angle, expand=True, resample=Image.Resampling.BICUBIC, fillcolor=(0, 0, 0, 0))
    return layer

# EDIT 4 · 시작값은 모두 같으므로 완료 조건을 통과하지 않습니다.
angles = [0, 0, 0]
alphas = [255, 255, 255]
positions = [(300, 300), (300, 300), (300, 300)]

# DO NOT EDIT · 레이어 생성과 합성
layers = []
for target_size, angle, alpha in zip(LAYER_SIZES, angles, alphas):
    layer = make_layer(working_image, crop_box, target_size, angle, alpha)
    layers.append(layer)

canvas = Image.new("RGBA", (1000, 1000), (29, 33, 31, 255))
for layer, position in zip(layers, positions):
    canvas.alpha_composite(layer, dest=position)

EXECUTION_ORDER.append(4)
print("각도:", angles)
print("알파:", alphas)
print("위치:", positions)
print("실제 레이어 크기:", [layer.size for layer in layers])
display(canvas)

## STEP 4 · PNG 저장
값을 마지막으로 수정한 뒤 STEP 3을 다시 실행하고, 이어서 이 저장 셀을 실행합니다.

In [ ]:
# DO NOT EDIT · 최종 PNG 저장
canvas.save(output_filename)
EXECUTION_ORDER.append(5)
print("저장 완료:", output_filename)
display(Image.open(output_filename))

## FINAL CHECK · 새 세션에서 모두 실행
런타임을 다시 시작한 뒤 **모두 실행**합니다. 오류가 나면 마지막 한글 안내를 읽고 해당 `EDIT` 값만 고칩니다. 이 셀은 수정하지 않습니다.

In [ ]:
# DO NOT EDIT · FINAL CHECK
EXECUTION_ORDER.append(6)
assert tuple(EXECUTION_ORDER) == (1, 2, 3, 4, 5, 6), "새 세션에서 위에서 아래로 모두 실행하세요."

assert student_id.strip() and "입력" not in student_id, "학번을 실제 값으로 바꾸세요."
assert student_name.strip() and "입력" not in student_name, "이름 또는 제출 확인이 가능한 별명을 작성하세요."
assert len(composition_intent.strip()) >= 20 and "작성하세요" not in composition_intent, "구성 의도를 20자 이상의 완전한 문장으로 작성하세요."
expected_filename = f"week07_{student_id}_{student_name}.png"
assert output_filename == expected_filename, f"PNG 파일명은 {expected_filename}이어야 합니다."

assert source_choice in {"provided", "own"}, "source_choice는 provided 또는 own입니다."
for field_name, field_value in {
    "제목": source_title,
    "창작자": source_creator,
    "원본 주소": source_url,
    "라이선스": source_license,
    "변형 내용": change_description,
}.items():
    assert len(field_value.strip()) >= 5, f"{field_name} 기록을 다섯 글자 이상 작성하세요."
assert Path(source_path).read_bytes() == source_bytes_before, "원본 파일이 바뀌었습니다. 원본을 다시 준비하세요."
assert Path(source_path).resolve() != Path(output_filename).resolve(), "원본과 결과물 파일명을 다르게 작성하세요."

left, top, right, bottom = crop_box
assert 0 <= left < right <= source_image.width, "crop_box의 left와 right를 원본 너비 안에서 확인하세요."
assert 0 <= top < bottom <= source_image.height, "crop_box의 top과 bottom을 원본 높이 안에서 확인하세요."
assert (right - left, bottom - top) != source_image.size, "원본 전체보다 작은 영역을 자르세요."

assert len(layers) >= 3, "변형 레이어를 세 개 이상 만드세요."
assert len(angles) == len(alphas) == len(positions) == len(LAYER_SIZES) == 3, "각도·알파·위치는 각각 세 개여야 합니다."
assert all(isinstance(angle, (int, float)) and -30 <= angle <= 30 for angle in angles), "각도는 -30도부터 30도 사이의 숫자로 작성하세요."
assert sum(angle != 0 for angle in angles) >= 2, "0이 아닌 회전 각도를 두 개 이상 사용하세요."
assert any(angle < 0 for angle in angles) and any(angle > 0 for angle in angles), "음수 각도와 양수 각도를 모두 사용하세요."
assert all(isinstance(alpha, int) and 100 <= alpha <= 240 for alpha in alphas), "알파 값은 100부터 240 사이의 정수로 작성하세요."
assert len(set(alphas)) >= 2, "서로 다른 알파 값을 최소 두 가지 사용하세요."
assert len(set(positions)) == 3, "세 레이어의 위치를 서로 다르게 작성하세요."
for layer, position in zip(layers, positions):
    assert isinstance(position, tuple) and len(position) == 2, "각 위치는 (x, y) 튜플이어야 합니다."
    x, y = position
    assert isinstance(x, int) and isinstance(y, int), "위치의 x와 y는 정수여야 합니다."
    assert 0 <= x and 0 <= y and x + layer.width <= 1000 and y + layer.height <= 1000, "각 레이어가 1000 × 1000 캔버스 안에 들어오도록 위치를 조정하세요."

assert canvas.mode == "RGBA" and canvas.size == (1000, 1000), "캔버스는 1000 × 1000 RGBA여야 합니다."
assert Path(output_filename).exists(), "PNG 저장 셀을 실행하세요."
checked_file = Image.open(output_filename).convert("RGBA")
assert checked_file.size == canvas.size, "저장된 PNG 크기를 확인하세요."
assert checked_file.tobytes() == canvas.tobytes(), "PNG가 최신 상태가 아닙니다. STEP 3과 STEP 4를 다시 실행하세요."

print("✅ 제출 정보와 구성 의도 검사 통과")
print("✅ 제목·창작자·원본 주소·라이선스·변형 내용 기록 통과")
print("✅ 자르기·크기 변경·회전 레이어 세 개 검사 통과")
print("✅ 1000 × 1000 RGBA 합성과 PNG 최신 상태 검사 통과")
print("🎉 WEEK 07 TRANSFORMATION PROTOTYPE COMPLETE")